# Step 1 — USAD replication on SMD (COMP8240 Update)
Runs the **official USAD code** on several SMD machines and reports the paper's metrics:
Precision, Recall, F1 (with and without point adjustment) and F1*.

**Before running:** Runtime → Change runtime type → **T4 GPU**. Then Runtime → **Run all**.

In [ ]:
# 1. Get the official USAD code and the SMD dataset
!git clone -q https://github.com/manigalati/usad.git
!git clone -q --depth 1 https://github.com/NetManAIOps/OmniAnomaly.git
import sys
sys.path.append('/content/usad')

In [ ]:
# 2. Imports
import time
import numpy as np
import pandas as pd
import torch
import torch.utils.data as data_utils
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import precision_recall_fscore_support, roc_auc_score

from usad import *                      # official model: UsadModel, training, testing
from utils import get_default_device, to_device
device = get_default_device()
print('Device:', device)

In [ ]:
# 3. Settings (same as the official notebook / your proposal run)
SMD_DIR     = '/content/OmniAnomaly/ServerMachineDataset'
MACHINES    = ['machine-1-1', 'machine-2-1', 'machine-3-1']   # one machine from each of SMD's 3 groups
WINDOW_SIZE = 12
HIDDEN_SIZE = 100
BATCH_SIZE  = 7919
N_EPOCHS    = 100
torch.manual_seed(0); np.random.seed(0)

In [ ]:
# 4. Helper functions
def load_machine(name):
    train  = np.loadtxt(f'{SMD_DIR}/train/{name}.txt', delimiter=',')
    test   = np.loadtxt(f'{SMD_DIR}/test/{name}.txt', delimiter=',')
    labels = np.loadtxt(f'{SMD_DIR}/test_label/{name}.txt', delimiter=',').astype(int)
    scaler = MinMaxScaler().fit(train)              # fit on normal training data only
    return scaler.transform(train), scaler.transform(test), labels

def make_windows(x, w):
    idx = np.arange(w)[None, :] + np.arange(x.shape[0] - w)[:, None]
    return x[idx]

def point_adjust(pred, label):
    # If any point in a true anomaly segment is flagged, count the whole segment as detected
    pred = pred.copy(); i, n = 0, len(label)
    while i < n:
        if label[i] == 1:
            j = i
            while j < n and label[j] == 1:
                j += 1
            if pred[i:j].any():
                pred[i:j] = 1
            i = j
        else:
            i += 1
    return pred

def best_f1(score, label, adjust):
    # Search thresholds and keep the one with best F1 (OmniAnomaly benchmark protocol)
    best = (0, 0, 0, 0)
    for th in np.quantile(score, np.linspace(0.5, 0.9999, 300)):
        pred = (score >= th).astype(int)
        if adjust:
            pred = point_adjust(pred, label)
        p, r, f, _ = precision_recall_fscore_support(label, pred, average='binary', zero_division=0)
        if f > best[2]:
            best = (p, r, f, th)
    return best

In [ ]:
# 5. Train + test USAD on one machine
def run_machine(name):
    train, test, labels = load_machine(name)
    n_feat = train.shape[1]
    w_train, w_test = make_windows(train, WINDOW_SIZE), make_windows(test, WINDOW_SIZE)
    win_labels = np.array([1 if labels[i:i+WINDOW_SIZE].sum() > 0 else 0 for i in range(len(w_test))])

    w_size, z_size = WINDOW_SIZE * n_feat, WINDOW_SIZE * HIDDEN_SIZE
    split = int(0.8 * len(w_train))
    to_t = lambda a: torch.from_numpy(a).float().view(-1, w_size)
    train_loader = data_utils.DataLoader(data_utils.TensorDataset(to_t(w_train[:split])), batch_size=BATCH_SIZE, shuffle=False)
    val_loader   = data_utils.DataLoader(data_utils.TensorDataset(to_t(w_train[split:])), batch_size=BATCH_SIZE, shuffle=False)
    test_loader  = data_utils.DataLoader(data_utils.TensorDataset(to_t(w_test)), batch_size=BATCH_SIZE, shuffle=False)

    model = to_device(UsadModel(w_size, z_size), device)
    t0 = time.time()
    history = training(N_EPOCHS, model, train_loader, val_loader)
    train_min = (time.time() - t0) / 60

    with torch.no_grad():
        results = testing(model, test_loader)
    score = torch.cat([r.flatten() for r in results]).cpu().numpy()

    raw = best_f1(score, win_labels, adjust=False)
    pa  = best_f1(score, win_labels, adjust=True)
    np.save(f'{name}_score.npy', score); np.save(f'{name}_labels.npy', win_labels)
    return {
        'machine': name, 'train_len': len(train), 'test_len': len(test), 'channels': n_feat,
        'anomaly_%': round(100 * labels.mean(), 2), 'train_min': round(train_min, 1),
        'AUC': roc_auc_score(win_labels, score),
        'P_raw': raw[0], 'R_raw': raw[1], 'F1_raw': raw[2],
        'P_PA': pa[0],  'R_PA': pa[1],  'F1_PA': pa[2], 'th_PA': pa[3],
    }, history

In [ ]:
# 6. Run all machines (this is the long cell)
rows, histories = [], {}
for m in MACHINES:
    print(f'\n===== {m} =====')
    row, hist = run_machine(m)
    rows.append(row); histories[m] = hist

df = pd.DataFrame(rows)
df.to_csv('smd_results.csv', index=False)
pd.set_option('display.precision', 4)
df

In [ ]:
# 7. Summary in the paper's format (F1* = F1 of the AVERAGE precision and recall)
summary = []
for tag in ['PA', 'raw']:
    P, R = df[f'P_{tag}'].mean(), df[f'R_{tag}'].mean()
    summary.append({'setting': 'with point adjustment' if tag == 'PA' else 'without point adjustment',
                    'Precision': P, 'Recall': R,
                    'F1 (mean of machines)': df[f'F1_{tag}'].mean(),
                    'F1*': 2 * P * R / (P + R)})
summary = pd.DataFrame(summary)
summary.to_csv('smd_summary.csv', index=False)
summary

In [ ]:
# 8. Video figures: training losses + anomaly score vs true anomalies
m = MACHINES[0]
h = histories[m]
plt.figure(figsize=(8, 4))
plt.plot([x['val_loss1'] for x in h], label='loss1 (AE1)')
plt.plot([x['val_loss2'] for x in h], label='loss2 (AE2)')
plt.xlabel('epoch'); plt.ylabel('validation loss'); plt.title(f'USAD training losses - {m}')
plt.legend(); plt.grid(alpha=.3); plt.tight_layout(); plt.savefig('fig_losses.png', dpi=150); plt.show()

score, lab = np.load(f'{m}_score.npy'), np.load(f'{m}_labels.npy')
th = df.loc[df.machine == m, 'th_PA'].values[0]
plt.figure(figsize=(12, 4))
plt.plot(score, lw=.6, label='USAD anomaly score')
plt.fill_between(range(len(lab)), 0, score.max(), where=lab == 1, color='red', alpha=.2, label='true anomaly')
plt.axhline(th, color='k', ls='--', lw=1, label='threshold')
plt.xlabel('test window'); plt.ylabel('score'); plt.title(f'USAD anomaly score on SMD {m}')
plt.legend(loc='upper right'); plt.tight_layout(); plt.savefig('fig_scores.png', dpi=150); plt.show()

In [ ]:
# 9. Download everything you need for the slides
from google.colab import files
for f in ['smd_results.csv', 'smd_summary.csv', 'fig_losses.png', 'fig_scores.png']:
    files.download(f)